# import

In [2]:
import pandas as pd
import json
import os

# Créer le dossier processed s'il n'existe pas
os.makedirs("../dataset/processed", exist_ok=True)

print("✅ Importations OK")

✅ Importations OK


# 2: Charger le CSV nettoyé

In [3]:
df = pd.read_csv("../dataset/processed/resume_jd_match_cleaned.csv")
print(f"Chargé: {len(df)} exemples")
print("\nDistribution des labels avant équilibrage:")
print(df["label"].value_counts())

Chargé: 6240 exemples

Distribution des labels avant équilibrage:
label
No Fit           3142
Potential Fit    1556
Good Fit         1542
Name: count, dtype: int64


# 3: Équilibrer les labels

In [4]:
min_count = df["label"].value_counts().min()
print(f"Nombre minimum par label: {min_count}")

df_balanced = df.groupby("label").head(min_count)

print("\nDistribution des labels après équilibrage:")
print(df_balanced["label"].value_counts())
print(f"\nTotal après équilibrage: {len(df_balanced)} exemples")

Nombre minimum par label: 1542

Distribution des labels après équilibrage:
label
No Fit           1542
Potential Fit    1542
Good Fit         1542
Name: count, dtype: int64

Total après équilibrage: 4626 exemples


# 4: Fonction de conversion label -> analyse détaillée

In [5]:
def label_to_analysis(label):
    """Convertit le label en analyse RH détaillée"""
    
    if label == "Good Fit":
        score = 85
        points_forts = "Compétences techniques alignées, Expérience pertinente"
        points_faibles = "À confirmer en entretien"
        verdict = "Bon candidat, recommandé pour entretien"
    elif label == "Potential Fit":
        score = 60
        points_forts = "Base solide, Potentiel de développement"
        points_faibles = "Manque certaines compétences clés"
        verdict = "Candidat potentiel, nécessite formation complémentaire"
    else:  # "No Fit"
        score = 25
        points_forts = "Points transférables limités"
        points_faibles = "Profil non adapté au poste"
        verdict = "Candidat non recommandé pour ce poste"
    
    return f"""SCORE: {score}%

POINTS FORTS:
- {points_forts}

POINTS FAIBLES:
- {points_faibles}

VERDICT:
{verdict}"""

# Tester la fonction
print("Test conversion Good Fit:")
print(label_to_analysis("Good Fit"))

Test conversion Good Fit:
SCORE: 85%

POINTS FORTS:
- Compétences techniques alignées, Expérience pertinente

POINTS FAIBLES:
- À confirmer en entretien

VERDICT:
Bon candidat, recommandé pour entretien


# 5: Transformer en format conversationnel JSONL

In [6]:
output_file = "../dataset/processed/conversations.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for idx, row in df_balanced.iterrows():
        conv = {
            "messages": [
                {
                    "role": "system",
                    "content": "Tu es un expert RH spécialisé dans l'analyse de CV. Pour chaque offre d'emploi et CV, tu dois fournir un score (0-100), les points forts, les points faibles, et un verdict."
                },
                {
                    "role": "user",
                    "content": row["text_clean"][:2000]
                },
                {
                    "role": "assistant",
                    "content": label_to_analysis(row["label"])
                }
            ]
        }
        f.write(json.dumps(conv, ensure_ascii=False) + "\n")

print(f"✅ Sauvegardé: {len(df_balanced)} exemples")
print(f"📁 Fichier: {output_file}")

✅ Sauvegardé: 4626 exemples
📁 Fichier: ../dataset/processed/conversations.jsonl


# 6: Vérifier le premier exemple

In [7]:
with open("../dataset/processed/conversations.jsonl", "r", encoding="utf-8") as f:
    first = json.loads(f.readline())

print("=" * 50)
print("SYSTEM:")
print(first["messages"][0]["content"][:200])
print("\n" + "=" * 50)
print("USER (début):")
print(first["messages"][1]["content"][:300])
print("\n" + "=" * 50)
print("ASSISTANT:")
print(first["messages"][2]["content"])
print("\n" + "=" * 50)
print("✅ Format JSONL valide")

SYSTEM:
Tu es un expert RH spécialisé dans l'analyse de CV. Pour chaque offre d'emploi et CV, tu dois fournir un score (0-100), les points forts, les points faibles, et un verdict.

USER (début):
For the given job description <<Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally, with over 30+ locations in the US and global operations in 32 coun

ASSISTANT:
SCORE: 25%

POINTS FORTS:
- Points transférables limités

POINTS FAIBLES:
- Profil non adapté au poste

VERDICT:
Candidat non recommandé pour ce poste

✅ Format JSONL valide


# 7: Statistiques finales

In [8]:
print("=" * 50)
print("📊 RÉSUMÉ FINAL DU DATASET")
print("=" * 50)
print(f"Total exemples: {len(df_balanced)}")
print(f"\nDistribution:")
print(df_balanced["label"].value_counts())
print(f"\nFichier de sortie: ../dataset/processed/conversations.jsonl")
print("✅ Prêt pour le fine-tuning QLoRA !")

📊 RÉSUMÉ FINAL DU DATASET
Total exemples: 4626

Distribution:
label
No Fit           1542
Potential Fit    1542
Good Fit         1542
Name: count, dtype: int64

Fichier de sortie: ../dataset/processed/conversations.jsonl
✅ Prêt pour le fine-tuning QLoRA !
